### Data Loader
[인공지능 기술·산업 생태계 육성방안
연구](https://spri.kr/posts/view/23669)를 이용하여 진행한다.



In [1]:
from common import neo4j_client
from settings import get_settings
from common.embedding_client import EmbeddingClient
from common.llm_client import LLMClient
from common.neo4j_client import Neo4jCustomClient

llm = LLMClient()
neo4j_client = Neo4jCustomClient()
embeddings = EmbeddingClient()


bolt://localhost:7687


In [2]:
print(f"LLM model {llm.get_model_name()}")
print(f"Embedding model {embeddings.get_model_name()}")
print(f"Neo4j Connection {neo4j_client.verify_connectivity()}")

LLM model google/gemma-4-e4b
Embedding model text-embedding-bge-m3
Neo4j Connection True


설정

In [3]:
import pdfplumber
from pathlib import Path

FILE_PATH = Path.cwd().parent/"data"/"RE-185. 인공지능 기술·산업 생태계 육성방안 연구.pdf"

text = ""

with pdfplumber.open(FILE_PATH) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

In [4]:
text[:50]
print(len(text))

146476


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=40,
)

chunks = text_splitter.split_text(text)
print(len(chunks))
print(chunks[0])

314
연구보고서 RE-185
인공지능 기술·산업 생태계 육성방안
연구
A Study on the Promotion Policy for the Korean Technological and
Industirial Ecosystem in Artificial Intelligence
봉강호 / 안성원
2025. 4.이 보고서는 2024년도 과학기술정보통신부 정보통신진흥기금을 지원
받아 수행한 연구결과로 보고서 내용은 연구자의 견해이며, 과학기술정보
통신부의 공식입장과 다를 수 있습니다.목 차
제1장 서론 ·························································································································· 1


### PDF Text Preprocessing 1
PDF에서 추출한 텍스트에는 목차의 점선 리더(dot leader)가 포함되어 있다.
```text
제1장 서론 ··········································· 1
```
이러한 문자열은 문서의 의미를 가지지 않으며, 청킹 및 임베딩 시 불필요한 토큰을 증가시키고 검색 품질을 저하시킬 수 있다.
따라서 정규식을 이용하여 연속된 점선(`···`, `...`)을 제거하는 전처리를 수행한다.


In [6]:
import re

def clean_pdf_text(text: str) -> str:
    text = re.sub(r"[·.]{3,}", " ", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [8]:
clean_text = clean_pdf_text(text)
chunks = text_splitter.split_text(clean_text)
print(len(chunks))
print(chunks[0])
print(type(chunks[0]))

297
연구보고서 RE-185
인공지능 기술·산업 생태계 육성방안
연구
A Study on the Promotion Policy for the Korean Technological and
Industirial Ecosystem in Artificial Intelligence
봉강호 / 안성원
2025. 4.이 보고서는 2024년도 과학기술정보통신부 정보통신진흥기금을 지원
받아 수행한 연구결과로 보고서 내용은 연구자의 견해이며, 과학기술정보
통신부의 공식입장과 다를 수 있습니다.목 차
제1장 서론 1
제1절 연구 배경 및 필요성 1
제2절 연구 목적 및 내용 5
제2장 주요국 정책 추진 현황 분석 6
제1절 국가별 현황 6
1. 미국 6
2. 중국 14
3. EU 23
4. 싱가포르 30
제2절 소결 및 시사점 38
제3장 글로벌 AI 연구 현황 분석 42
제1절 데이터 수집 방법론 개발 필요성 42
제2절 데이터 수집 방법론 43
1. AI 기술 키워드 도출 및 검색식 작성 43
<class 'str'>


### PDF Text Preprocessing 2

PDF 텍스트에서 점선 리더(dot leader)를 제거한 뒤에도 다음과 같은 목차 항목이 남아있다.
```text
제1절 데이터 수집 방법론 개발 필요성 42
```
- 위의 마지막 숫자 42는 페이지 번호이며, 전체 문장은 본문이 아니라 목차의 항목에 해당한다.
- 페이지 번호만 제거하면 다음과 같이 목차 제목이 여전히 청크에 포함된다.
```text
제1절 데이터 수집 방법론 개발 필요성
```
목차 항목은 실제 본문 내용이 아니며, 벡터 검색 시 동일한 제목이 본문보다 우선 검색되거나 불필요한 청크가 생성될 수 가능성이 있다.
1. 페이지 번호만 제거하지 않고, 점선 리더와 페이지 번호로 구성된 목차 항목은 해당 줄 전체를 제거?
2. 목차 항목을 정규식으로 개별 제거하지 않고, 두 번째 목차가 포함된 페이지 전체를 문서 처리 대상에서 제외?

```
text = re.sub(
    r"^.*[·.]{3,}\s*\d+\s*$",
    "",
    text,
    flags=re.MULTILINE,
)
```

In [9]:
print(len(chunks))

297


In [10]:
vectors = embeddings.embed_documents(chunks)

print(vectors[0][:3])
print(len(vectors))
print(len(vectors[0]))

[-0.048613667488098145, 0.015080631710588932, -0.01646498404443264]
297
1024


Neo4j Driver 연결한다.

In [11]:

# 기존와  노드와 인덱스를 제거한다.
query = "000-drop-node"
neo4j_client.execute_query(query, database ="neo4j")
query = "000-drop-index"
neo4j_client.execute_query(query, database ="neo4j")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x16592b2c0>, keys=[])

In [12]:
# Neo4J Index를 생성한다.
query= "001-create-vector-index"
neo4j_client.execute_query(query, database_="neo4j")


EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x165974b00>, keys=[])

In [13]:
query = "002-show-index"
records, summary, keys = neo4j_client.execute_query(query, database_="neo4j")

In [14]:
records

[<Record name='pdf' state='ONLINE' labelsOrTypes=['Chunk'] properties=['embedding'] type='VECTOR'>]

In [15]:
summary

In [16]:
# 레코드 확인
for record in records:
    print(record.data())

{'name': 'pdf', 'state': 'ONLINE', 'labelsOrTypes': ['Chunk'], 'properties': ['embedding'], 'type': 'VECTOR'}


Neo4j 저장

In [17]:
query = "003-create-chunk-node"

In [18]:
embedding_vectors = embeddings.embed_documents(chunks)

In [19]:
neo4j_client.execute_query(query, chunks=chunks, embeddings=embedding_vectors)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x165976060>, keys=[])

In [20]:
query =  "004-match-all"
records, _, _ = neo4j_client.execute_query(query)

print(records[0]["c.text"][0:30])
print(records[0]["c.embedding"][0:3])

연구보고서 RE-185
인공지능 기술·산업 생태계 육성
[-0.048613667488098145, 0.015080631710588932, -0.01646498404443264]


In [21]:
question = "인공지능 산업 생태계 육성 방안의 목적은 무엇인가?"
question_embedding = embeddings.embed_query(question)
print(len(question_embedding))

1024


In [22]:
query = "005-query-vector-index"
similar_records, _, _ = neo4j_client.execute_query_file(query, question_embedding=question_embedding, k=4)

# similar_records, summary, keys = neo4j_client.execute_query_file(
#     "005-query-vector-index",
#     question_embedding=question_embedding,
#     k=4,
# )

# print(keys)


for i, record in enumerate(similar_records, start=1):
    print(f"{i} {'-' * 20}")
    print(record["text"])
    print(f"score: {record["score"]}, index: {record["index"]}")
    print("-"*22)

1 --------------------
구분 NAIS 1.0(2019년) NAIS 2.0(2022년)
국가 경제 및 사회를 발전시키는 데 AI 생태계를 고도화하고 경제사회 전반에
목표
AI의 잠재력을 활용 AI를 내재화하여 지속가능한 발전 도모
정책 § AI 활용 기회 모색 § AI를 생존에 필수적인 요소로 인식
기조 § 프로젝트(파일럿)를 통한 시범 운영 § 스케일업(규모 확대) 및 글로벌화
§ 정부 주도 시범사업 추진 § 장기적·거시적 관점으로 생태계 구축
방식
§ 기초 인프라 및 거버넌스 구축 § 민-관 및 글로벌 협력 확대
§ 머신러닝, 컴퓨터비전 등 기초 AI R&D § 첨단 AI(생성형 AI, 자율주행, 로보틱스
주요 지원 및 데이터 거버넌스 마련 등) R&D 투자 및 대규모 컴퓨팅 인프라
추진 § 대학·연구기관 전문가 육성 및 재교육 확대 구축
내용 § AI 윤리·투명성 체계 토대 마련(Model § 석·박사 인재 육성 및 해외 고급인력 유치
score: 0.8086662292480469, index: 95
----------------------
2 --------------------
글로벌 시장 진출을 지원하는 정책적 노력을 더욱 확대할 필요가 있다.
마지막으로, 국내 AI 생태계가 선순환하고 자생할 수 있도록 AI 확산을 위한 공격적인
정책 추진을 통해 AI 수요를 확대해야 한다. 이를 위해 공공부문에서 국내 기업의 AI
솔루션/서비스를 우선 구매하는 제도뿐 아니라, 이러한 제도의 적용을 받지 아니하는
공공·민간 구매자에게 국내 기업의 AI 솔루션/서비스를 구매함에 따른 기회비용을 상쇄
할 만큼 충분한 인센티브를 제공하는 방안을 도입할 필요가 있다.
④ AI 기반 인프라
AI 기반 인프라 측면에서는 대규모 컴퓨팅 인프라와 함께 산업 수요에 부합하는 AI
학습용 데이터를 제공하는 것이 강조된다는 점이다. AI 분야에서 컴퓨팅 인프라와 데이
터의 중요성은 아무리 강조해도 지나치지 않는다. 그러나 데이터와 관련하여, 산업 현장
에서 필요로 하거

In [23]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 AI 기술 및 산업 정책 연구 전문가입니다.

다음 규칙을 반드시 지켜 답변하세요.
- 제공된 문서 내용만 근거로 답변합니다.
- 문서에 없는 내용은 추측하거나 임의로 생성하지 않습니다.
- 답을 찾을 수 없는 경우 "제공된 문서에서 해당 내용을 찾을 수 없습니다."라고 답변합니다.
- 답변은 명확하고 간결하게 작성합니다.
            """.strip(),
        ),
        (
            "human",
            """
다음은 검색된 문서입니다.

{context}

---

위 문서에 포함된 정보만 사용하여 아래 질문에 답변하세요.

질문:
{question}
            """.strip(),
        ),
    ]
)

In [24]:
similar_records, _, _ = neo4j_client.execute_query(query, question_embedding=question_embedding, k=4)

In [26]:
context = "\n\n".join(doc["text"] for doc in similar_records)
#question = "인공지능 산업 생태계 육성의 목적은 무엇인가?"
question = "보고서에서 중요하게 다루는 AI 핵심 기술은 무엇인가?"
messages = prompt.invoke(
    {
        "context": context,
        "question": question,
    }
)

response = llm.chat(messages)

print(response)

보고서에서 중요하게 다루는 AI 핵심 기술은 다음과 같습니다.

*   **기초 AI R&D:** 머신러닝, 컴퓨터비전
*   **첨단 AI:** 생성형 AI, 자율주행, 로보틱스


In [36]:
from pathlib import Path
from langchain_core.prompts import ChatPromptTemplate

query = "005-query-vector-index"

# 2. RAG 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 AI 기술 및 산업 정책 연구 전문가입니다.

다음 규칙을 반드시 지켜 답변하세요.
- 제공된 문서 내용만 근거로 답변하세요.
- 문서에 없는 내용은 추측하지 마세요.
- 답을 찾을 수 없다면
  "제공된 문서에서 해당 내용을 찾을 수 없습니다."라고 답변하세요.
- 답변은 한국어로 명확하게 작성하세요.
            """.strip(),
        ),
        (
            "human",
            """
다음은 질문과 관련하여 검색된 문서입니다.

{context}

---

위 문서에 포함된 정보만 사용하여 아래 질문에 답변하세요.

질문:
{question}
            """.strip(),
        ),
    ]
)


# 3. 질문부터 LLM 답변까지 한 번에 처리
def ask_rag(question: str, k: int = 4) -> str:
    question_embedding = embeddings.embed_query(question)


    similar_records, _, _ = neo4j_client.execute_query(
        query,
        question_embedding=question_embedding,
        k=k,
        database_="neo4j",
    )

    if not similar_records:
        return "관련 문서를 찾을 수 없습니다."

    context = "\n\n".join(
        f"[문서 {i + 1} | 유사도: {record['score']:.4f}]\n"
        f"{record['text']}"
        for i, record in enumerate(similar_records)
    )

    messages = prompt.invoke(
        {
            "context": context,
            "question": question,
        }
    )

    response = llm.chat(messages)

    return response

In [37]:
question = "AI 전문 인력 양성을 위해 어떤 정책을 제안하고 있는가?"

print(ask_rag(question, k=5))

제공된 문서에는 AI 전문 인력 양성을 위해 다음과 같은 정책적 접근과 구체적인 방안들이 제안되어 있습니다.

**1. 전략 및 시스템적 접근 강화**
*   **통합적이고 시스템적인 관점 지향:** 현재 인재 확보 노력과 미래 인재 양성 노력이 유기적으로 연계될 수 있는 '시스템적' 접근을 취하고, AI 인재 관련 정책의 중요도를 대폭 상향 조정해야 합니다.
*   **정책 컨트롤타워 강화:** AI 정책 기능을 통합하고 정책 컨트롤타워의 역할 및 권한을 더욱 강화할 필요가 있습니다.
*   **장기적 관점 확보:** 인재 정책은 단기간에 육성될 수 없으므로, 현재와 미래를 모두 고려하는 장기적 관점의 전략과 전폭적인 지원이 필요합니다.

**2. 인재 양성 및 육성 방안**
*   **생애주기별 지원:** 잠재력 있는 인재가 다양한 교육 및 실무 경험을 축적하며 장기간에 걸쳐 성장할 수 있도록 생애주기별로 지원하는 방안을 모색해야 합니다.
*   **교육 혁신:** 초·중등·대학 교육 혁신을 추진하고, 차세대 인재 육성을 위한 컴퓨터 과학 및 STEM 교육을 강화해야 합니다.
*   **현장 인력 지원:** 현재 산업 현장의 숙련 수요를 충족시키고, 도메인 전문가의 경력 전환을 위한 교육 및 지원을 확대해야 합니다.
*   **창업 및 연구 활동 지원:** 국내 AI 산업 생태계에서 도전과 혁신을 통해 새로운 가치를 창출할 기업가의 탄생을 유도하고, AI 인력의 연구 활동 및 창업을 지원해야 합니다.

**3. 글로벌 고급 인재 유치 노력**
*   **해외 인재 유치:** 해외 고급 인재를 유치하고, 우수 한인 AI 인재가 국내로 복귀하고 정착할 수 있도록 파격적인 지원을 제공하는 사업을 추진해야 합니다.
*   **국제적 모델 참고:** 주요국들의 사례처럼, AI 연구 활동 지원 및 해외 인재 유치 노력을 통해 자국 역량을 보강하는 것이 중요합니다.

**4. 정책적 투자 영역**
*   **시장 실패 영역에 대한 과감한 투자:** 기초 연구, AI 인